<a href="https://colab.research.google.com/github/Aarush713757/gbm-transcriptomics-research/blob/main/gbm_transcriptomics_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# =====================================================================
# PROJECT: gbm-transcriptomics-research
# PURPOSE: Rebuild pipeline for glioblastoma data analysis
# =====================================================================
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print("[System] Loading computational frameworks...")

# ---------------------------------------------------------------------
# MODULE 1: Data Acquisition & Preprocessing
# ---------------------------------------------------------------------
def preprocess_expression_data(matrix_data):
    """Handles data structure mapping rows/genes & normalizes counts."""
    print("[Module 1] Preprocessing transcriptomics matrix...")
    cleaned = pd.DataFrame(matrix_data).dropna(how='all').fillna(0)
    return np.log2(cleaned + 1)

# ---------------------------------------------------------------------
# MODULE 2: Differential Expression Analysis (DEA)
# ---------------------------------------------------------------------
def run_differential_expression(df, normal_idx, tumor_idx, alpha=0.05):
    """Finds statistically significant up/down-regulated genes."""
    print("[Module 2] Running Independent T-Tests across tissue cohorts...")
    dea_results = []
    for gene in df.index:
        normal_exp = df.loc[gene, normal_idx]
        tumor_exp = df.loc[gene, tumor_idx]
        log2_fc = np.mean(tumor_exp) - np.mean(normal_exp)
        _, p_val = stats.ttest_ind(normal_exp, tumor_exp, equal_var=False)
        dea_results.append({'Gene': gene, 'Log2FC': log2_fc, 'P_Value': p_val})

    res_df = pd.DataFrame(dea_results).set_index('Gene')
    significant = res_df[res_df['P_Value'] < alpha]
    print(f" -> Found {len(significant)} statistically significant genes.")
    return res_df

# ---------------------------------------------------------------------
# MODULE 3: Machine Learning & Predictive Modeling
# ---------------------------------------------------------------------
def train_predictive_model(df, target_labels):
    """Builds a predictive classification model to identify tissue type."""
    print("[Module 3] Initializing Random Forest Classifier...")
    X = df.T # Transpose matrix: samples become rows, genes become features
    y = target_labels
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f" -> Model Evaluation complete. Accuracy: {acc * 100:.2f}%")
    return clf

print("[System] Rebuilt baseline prototype successfully compiled and loaded!")

[System] Loading computational frameworks...
[System] Rebuilt baseline prototype successfully compiled and loaded!


In [9]:
# =====================================================================
# SIMULATED TESTING SCRIPT
# =====================================================================
# 1. Create a mock gene expression matrix (100 genes, 40 samples total)
genes = [f"Gene_{i}" for i in range(1, 101)]
normal_cols = [f"Normal_Sample_{i}" for i in range(1, 21)]
tumor_cols = [f"Tumor_Sample_{i}" for i in range(1, 21)]  # Fixed naming here
all_samples = normal_cols + tumor_cols

# Generate random base count data
np.random.seed(42)
mock_data = np.random.randint(10, 500, size=(100, 40))
raw_matrix = pd.DataFrame(mock_data, index=genes, columns=all_samples)  # Fixed 'columns' parameter

# Test module: Preprocessing
norm_matrix = preprocess_expression_data(raw_matrix)

# Test module: Differential expression analysis
dea_output = run_differential_expression(norm_matrix, normal_cols, tumor_cols)  # Fixed spelling of normal_cols

# Test module: Machine learning model
labels = [0]*20 + [1]*20
trained_model = train_predictive_model(norm_matrix, labels)

[Module 1] Preprocessing transcriptomics matrix...
[Module 2] Running Independent T-Tests across tissue cohorts...
 -> Found 4 statistically significant genes.
[Module 3] Initializing Random Forest Classifier...
 -> Model Evaluation complete. Accuracy: 37.50%
